### Why Integrate an AST Parser?
While our Machine Learning model analyzes *behavior* (transactions and wallet networks), the Abstract Syntax Tree (AST) parser analyzes the *structure* of the smart contract's actual code. Combining these creates a much more robust early warning system.

| Machine Learning (Tasks 1-3) | AST Parser (Task 4 Bonus) |
| :--- | :--- |
| Detects suspicious **behavior** | Detects suspicious **code logic** |
| Learns statistically from transaction graphs | Uses rule-based static analysis |
| Relies on historical on-chain patterns | Does not require transaction history |
| May miss hidden backdoors before they are used | Directly flags malicious functions (e.g., hidden mints) |

In [2]:
import json
import pandas as pd
from solidity_parser import parser
from catboost import CatBoostClassifier

### Rule-Based Vulnerability Detection
An Abstract Syntax Tree (AST) breaks down the Solidity source code into a searchable, hierarchical tree. We define specific rules to traverse this tree and hunt for classic "rug pull" red flags:
* **Hidden Mints:** Functions allowing developers to secretly create infinite tokens.
* **Dangerous Owner Privileges:** Admin controls that can drain liquidity or modify core parameters.
* **Blacklisting:** Logic that prevents specific users from selling their tokens (creating a honeypot).
* **Trading Control Functions:** Code that allows the owner to pause or disable trading entirely at their discretion.

Each finding is assigned a weighted penalty to calculate a structural risk score.

In [3]:
def detect_mint_function(ast):
    findings = []
    for child in ast.get('children', []):
        if child.get('type') == 'ContractDefinition':
            for subnode in child.get('subNodes', []):
                if subnode.get('type') == 'FunctionDefinition':
                    function_name = subnode.get('name', '')
                    if function_name and any(kw in function_name.lower() for kw in ['mint', 'generate', 'createtoken']):
                        findings.append({'issue': 'Hidden Mint Function', 'function': function_name, 'risk_score': 0.30})
    return findings

def detect_owner_privileges(ast):
    findings = []
    dangerous_keywords = ['withdraw', 'settax', 'blacklist', 'disable', 'lock', 'liquidity']
    for child in ast.get('children', []):
        if child.get('type') == 'ContractDefinition':
            for subnode in child.get('subNodes', []):
                if subnode.get('type') == 'FunctionDefinition':
                    function_name = subnode.get('name', '')
                    modifiers = subnode.get('modifiers', [])
                    has_only_owner = any(m.get('name') == 'onlyOwner' for m in modifiers)
                    
                    if has_only_owner and function_name and any(kw in function_name.lower() for kw in dangerous_keywords):
                        findings.append({'issue': 'Dangerous Owner Privilege', 'function': function_name, 'risk_score': 0.25})
    return findings

def detect_blacklist_logic(ast):
    findings = []
    for child in ast.get('children', []):
        if child.get('type') == 'ContractDefinition':
            for subnode in child.get('subNodes', []):
                if subnode.get('type') == 'StateVariableDeclaration':
                    variables = subnode.get('variables', [])
                    for variable in variables:
                        variable_name = variable.get('name', '')
                        if variable_name and any(kw in variable_name.lower() for kw in ['blacklist', 'blocked', 'banned']):
                            findings.append({'issue': 'Blacklist Mechanism Detected', 'variable': variable_name, 'risk_score': 0.20})
    return findings

def detect_trading_control(ast):
    findings = []
    for child in ast.get('children', []):
        if child.get('type') == 'ContractDefinition':
            for subnode in child.get('subNodes', []):
                if subnode.get('type') == 'FunctionDefinition':
                    function_name = subnode.get('name', '')
                    if function_name and any(kw in function_name.lower() for kw in ['pause', 'disabletrading', 'stoptrading']):
                        findings.append({'issue': 'Trading Control Function', 'function': function_name, 'risk_score': 0.20})
    return findings

def scan_contract(ast):
    """Combines all detection modules."""
    findings = []
    findings.extend(detect_mint_function(ast))
    findings.extend(detect_owner_privileges(ast))
    findings.extend(detect_blacklist_logic(ast))
    findings.extend(detect_trading_control(ast))
    return findings

def calculate_ast_risk(findings):
    total_score = sum(finding['risk_score'] for finding in findings)
    return min(round(total_score, 2), 1.0)

def generate_explanation(findings):
    explanations = []
    for finding in findings:
        issue = finding['issue']
        if 'Hidden Mint Function' in issue:
            explanations.append('Contract contains minting capability.')
        elif 'Dangerous Owner Privilege' in issue:
            explanations.append('Owner has dangerous administrative privileges.')
        elif 'Blacklist Mechanism' in issue:
            explanations.append('Blacklist or wallet restriction mechanism detected.')
        elif 'Trading Control' in issue:
            explanations.append('Trading control or pause function detected.')
        elif 'Unverified Contract' in issue:
            explanations.append('Source code is not publicly verified — a strong rug pull indicator.')
    return list(set(explanations))

### Autonomous Smart Contract Fetching
To make this a viable real-time tool, the system cannot rely on manually downloaded code. Instead, our auditing agent uses the **Etherscan V2 API** to autonomously fetch the live, uncompiled Solidity source code directly from the Ethereum mainnet using just the contract address. This allows on-the-fly auditing of any real-world token. If the contract source is not publicly verified, the system flags this as a fraud indicator in itself and applies a base structural penalty.

In [ ]:
import os
import requests
import re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent.parent / ".env")  # root .env, two levels above 03_Agents_and_Tools/
api_key = os.environ.get("ETHERSCAN_API_KEY", "")

# 1. Provide the target address
# contract_address = '0x21df3b628c2594c18c6f488d22b574f5a33e3fb4'
# contract_address = '0x27054b13b1b798b345b591a4d22e6562d47ea75a'
contract_address = '0x1f9840a85d5aF5bf1D1762F925BDADdC4201F984' # Uniswap (or a fraud address)

print(f"Fetching live source code for {contract_address}...")

# 2. Call Etherscan V2 API
url = f'https://api.etherscan.io/v2/api?chainid=1&module=contract&action=getsourcecode&address={contract_address}&apikey={api_key}'
response = requests.get(url)
data = response.json()

# 3. Process and Scan
if data.get('status') == '1' and data.get('result'):
    raw_source = data['result'][0].get('SourceCode', '')
    
    if not raw_source:
        # Contract exists but source is unverified — strong red flag itself
        print("WARNING: Contract source code is NOT verified on Etherscan.")
        print("Unverified contracts are a major fraud indicator.")
        results = [{'issue': 'Unverified Contract', 'function': 'N/A', 'risk_score': 0.30}]
        ast_risk_score = 0.30
        explanations = ["Source code is not publicly verified — a strong rug pull indicator."]
    
    else:
        # Source code exists — proceed with AST parsing as before
        if raw_source.startswith('{{'):
            raw_source = raw_source[1:-1]
            source_dict = json.loads(raw_source)
            source_code = "".join(
                file_data.get('content', '')
                for _, file_data in source_dict.get('sources', {}).items()
            )
        else:
            source_code = raw_source

        try:
            ast = parser.parse(source_code)
            results = scan_contract(ast)
            ast_risk_score = calculate_ast_risk(results)
            explanations = generate_explanation(results)
        except Exception as e:
            print(f"AST parsing failed: {e}. Falling back to string scan...")
        
        # Fallback: Basic string matching if the parser crashes
        results = []
        source_lower = source_code.lower()
        
        mint_matches = re.findall(r'function\s+(_?mint\w*)\s*\(', source_code, re.IGNORECASE)
        if mint_matches:
            for fn in mint_matches:
                results.append({'issue': 'Hidden Mint Function (Raw Match)', 'function': fn, 'risk_score': 0.30})

        owner_fns = re.findall(r'function\s+(\w+)\s*\([^)]*\)[^{]*onlyOwner', source_code, re.IGNORECASE)
        dangerous = ['withdraw', 'settax', 'blacklist', 'pause', 'disable', 'lock', 'liquidity']
        for fn in owner_fns:
            if any(kw in fn.lower() for kw in dangerous):
                results.append({'issue': 'Dangerous Owner Privilege (Raw Match)', 'function': fn, 'risk_score': 0.25})

        ast_risk_score = calculate_ast_risk(results) if results else 0.20
        explanations = generate_explanation(results) if results else ["Code parsing failed. Applied base structural penalty."]
    # --------------------------------------------

    print("-----------------------------------")
    print(f"AST Risk Score: {ast_risk_score}")
    print("Explanations:")
    if explanations:
        for exp in explanations:
            print(f"- {exp}")
    else:
        print("- No malicious capabilities detected.")

else:
    print("Contract not found on Etherscan.")
    results = []
    ast_risk_score = 0.30
    explanations = ["Contract not found. Applied base structural penalty."]


Fetching live source code for 0x1f9840a85d5aF5bf1D1762F925BDADdC4201F984...
-----------------------------------
AST Risk Score: 0.3
Explanations:
- Contract contains minting capability.


### Final Synthesis: ML + AST Integration
This is where the two pillars of our system combine. We load the trained CatBoost model (`rugpull_detector.cbm`) — using the same StandardScaler from Task 3 to ensure identical feature scaling — to get the statistical fraud probability based on on-chain network topology. Then, we apply a weighted formula to generate the definitive audited risk score:
* **70% Weight:** Machine Learning (On-chain behavioral data)
* **30% Weight:** AST Parser (Static code vulnerability data)

This ensures that even if a token hasn't started behaving suspiciously yet, hidden malicious code will still trigger an alert.

In [ ]:
import os
import joblib
from sklearn.preprocessing import StandardScaler

MODELS_DIR = os.path.join("..", "02_Model_Training", "exported_models")

# 1. Load the actual graph features generated by 01_Data_Pipeline/2_processing_and_graph.ipynb
print("Loading real network graph features from Task 2...")
df = pd.read_csv(os.path.join("..", "01_Data_Pipeline", "processed_features.csv"))
X_actual = df.drop("label", axis=1)

# uses the exact same scaling parameters as Task 3
scaler = joblib.load(os.path.join(MODELS_DIR, "scaler.joblib"))
X_scaled = scaler.transform(X_actual)  # transform only — no re-fitting

# Select a fraud-labelled row to represent the token we are auditing
fraud_rows = df[df['label'] == 1]
if len(fraud_rows) > 0:
    test_index = fraud_rows.index[0]
else:
    test_index = 0
print(f"Using row index {test_index} (label: {df['label'][test_index]})")
test_features = X_scaled[test_index].reshape(1, -1)

# 2. Load the machine learning model from 02_Model_Training/exported_models/
ml_model = CatBoostClassifier()
try:
    ml_model.load_model(os.path.join(MODELS_DIR, "rugpull_detector.cbm"))
    
    # Get the actual prediction using the REAL features
    ml_probability = ml_model.predict_proba(test_features)[:, 1][0] * 100
except:
    print("Model file not found. Using simulated ML probability for demonstration.")
    ml_probability = 72.0

print(f"Base ML Fraud Probability (from actual data): {ml_probability:.2f}%")

# 3. Combine Scores (70% ML, 30% AST)
final_risk_score = round((0.7 * ml_probability) + (0.3 * (ast_risk_score * 100)), 2)

def classify_risk(score):
    if score >= 75.0: return 'HIGH RISK'
    elif score >= 45.0: return 'MEDIUM RISK'
    return 'LOW RISK'

risk_level = classify_risk(final_risk_score)

# 4. Final JSON Output
final_output = {
    'ml_score': f"{ml_probability:.2f}%",
    'ast_score': f"{ast_risk_score * 100}%",
    'final_risk_score': f"{final_risk_score}%",
    'risk_level': risk_level,
    'findings': results
}

print("\n==================================================")
print("             FINAL AUDIT RESULT")
print("==================================================")
print(json.dumps(final_output, indent=2))


Loading real network graph features from Task 2...
Using row index 200 (label: 1)
Base ML Fraud Probability (from actual data): 85.74%

             FINAL AUDIT RESULT
{
  "ml_score": "85.74%",
  "ast_score": "30.0%",
  "final_risk_score": "69.02%",
  "risk_level": "MEDIUM RISK",
  "findings": [
    {
      "issue": "Hidden Mint Function (Raw Match)",
      "function": "mint",
      "risk_score": 0.3
    }
  ]
}


### Task 4 Summary & Project Outcome
By integrating the AST parser, we successfully elevated the Meme-Coin Early Warning System from a purely statistical tracker to a comprehensive security auditing tool. 

**Key Outcomes:**
1. **Proactive Detection:** While the ML model is highly sensitive to suspicious trading topologies, the AST parser can catch malicious contracts *before* any transactions even occur by directly reading the source code.
2. **Explainability:** The final JSON audit report provides users with concrete, readable explanations of *why* a token is dangerous (e.g., "Contract contains minting capability"), rather than just a black-box probability score.
3. **End-to-End Automation:** By leveraging the Etherscan API and linking the ML model with the AST parser, the pipeline is fully autonomous. It takes a single contract address, fetches live source code, and outputs a synthesized dual-layered risk assessment. Note: in this demonstration, the ML component uses a representative on-chain sample; a production deployment would fetch live transaction data per queried address.